### IDX Exchange Team **ds55**
### Beini Lan
# 05_advanced_models - Week 7 Gradient Boosting

This notebook evaluates advanced gradient-boosted tree models on the newest (2026 June)
completely unseen month of California single-family residence sales.

- **Latest training window:** June 2025 through May 2026
- **Untouched test month:** June 2026
- **Algorithms explored:** XGBoost and LightGBM
- **Reference:** the Week 6 Random Forest configuration, retrained on the same
  shifted data for a fair same-month comparison

The workflow applies the same Week 3 data preprocessing and Week 6 feature engineering to the shifted datasets, tunes only inside the
training period, and evaluates June 2026 exactly once at the end.

## Setup and reproducibility controls

The notebook uses fixed random seeds, complete-month chronological validation,
fold-fitted imputing/encoding, and the same leakage exclusions used in prior
modeling weeks. The package versions used for the executed notebook are pinned
in `W7 Advanced Models/requirements.txt`.

In [ ]:
from pathlib import Path
import gc
import json
import os
import random
import time
import warnings

import geopandas as gpd
import lightgbm as lgb
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    median_absolute_error,
    r2_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)


warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="lightgbm")
pd.set_option("display.max_columns", 160)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", "{:,.4f}".format)

TEAM = "Team ds55, Beini Lan"
RANDOM_STATE = 55
N_JOBS = -1
TRAIN_START_MONTH = "2025-06"
TRAIN_END_MONTH = "2026-05"
TEST_MONTH = "2026-06"
CV_VALIDATION_MONTHS = ["2026-03", "2026-04", "2026-05"]
FORBIDDEN_FEATURE_TERMS = [
    "ListPrice",
    "OriginalListPrice",
    "DaysOnMarket",
    "PurchaseContractDate",
    "ListingContractDate",
    "ContractStatusChangeDate",
    "PricePerLivingArea",
]

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)


def find_project_root():
    candidates = [Path.cwd(), *Path.cwd().parents]
    env_root = os.environ.get("IDX_PROJECT_ROOT")
    if env_root:
        candidates.insert(0, Path(env_root).expanduser())

    for candidate in candidates:
        if (
            (candidate / "raw data" / "CRMLSSold202606.csv").exists()
            and (candidate / "W6 Feature Engineering").exists()
        ):
            return candidate.resolve()
    raise FileNotFoundError(
        "Could not locate the IDX Exchange project root. "
        "Run from the project or set IDX_PROJECT_ROOT."
    )


PROJECT_ROOT = find_project_root()
RAW_DIR = PROJECT_ROOT / "raw data"
OUTPUT_DIR = PROJECT_ROOT / "W7 Advanced Models"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SCHOOL_DISTRICT_FILE = (
    PROJECT_ROOT
    / "W6 Feature Engineering"
    / "resources"
    / "California_School_District_Areas_2025-26.geojson"
)

environment_summary = pd.DataFrame(
    [
        {"item": "pandas", "value": pd.__version__},
        {"item": "scikit-learn", "value": __import__("sklearn").__version__},
        {"item": "XGBoost", "value": xgb.__version__},
        {"item": "LightGBM", "value": lgb.__version__},
        {"item": "random seed", "value": RANDOM_STATE},
    ]
)
display(environment_summary)

## Import the shifted monthly raw data into pandas

Exactly thirteen source files are required: twelve training months from
`CRMLSSold202506.csv` through `CRMLSSold202605.csv`, followed by the latest
holdout file `CRMLSSold202606.csv`. File-month and `CloseDate` month are checked
so a misnamed or mixed-period extract cannot silently enter the experiment.

In [2]:
RAW_COLUMNS = [
    "StateOrProvince",
    "PropertyType",
    "PropertySubType",
    "CloseDate",
    "ClosePrice",
    "Flooring",
    "Levels",
    "ViewYN",
    "PoolPrivateYN",
    "Latitude",
    "Longitude",
    "LivingArea",
    "AttachedGarageYN",
    "ParkingTotal",
    "YearBuilt",
    "BathroomsTotalInteger",
    "BedroomsTotal",
    "FireplaceYN",
    "MainLevelBedrooms",
    "NewConstructionYN",
    "GarageSpaces",
    "HighSchoolDistrict",
    "PostalCode",
    "AssociationFee",
    "LotSizeSquareFeet",
]
NUMERIC_RAW_COLUMNS = [
    "ClosePrice",
    "Latitude",
    "Longitude",
    "LivingArea",
    "ParkingTotal",
    "YearBuilt",
    "BathroomsTotalInteger",
    "BedroomsTotal",
    "MainLevelBedrooms",
    "GarageSpaces",
    "AssociationFee",
    "LotSizeSquareFeet",
]

expected_periods = pd.period_range(
    TRAIN_START_MONTH,
    TEST_MONTH,
    freq="M",
)
monthly_frames = []
source_rows = []

for period in expected_periods:
    source_month = period.strftime("%Y%m")
    source_path = RAW_DIR / f"CRMLSSold{source_month}.csv"
    if not source_path.exists():
        raise FileNotFoundError(f"Missing required raw file: {source_path}")

    header = pd.read_csv(source_path, nrows=0).columns.str.strip()
    missing_columns = sorted(set(RAW_COLUMNS).difference(header))
    if missing_columns:
        raise ValueError(f"{source_path.name} is missing columns: {missing_columns}")

    frame = pd.read_csv(
        source_path,
        usecols=RAW_COLUMNS,
        dtype={"PostalCode": "string"},
        low_memory=False,
    )
    frame.columns = frame.columns.str.strip()
    frame["SourceFile"] = source_path.name
    frame["SourceMonth"] = str(period)
    monthly_frames.append(frame)
    source_rows.append(
        {
            "source_file": source_path.name,
            "source_month": str(period),
            "raw_rows": len(frame),
        }
    )

raw = pd.concat(monthly_frames, ignore_index=True)
raw["CloseDate"] = pd.to_datetime(raw["CloseDate"], errors="coerce")
for column in NUMERIC_RAW_COLUMNS:
    raw[column] = pd.to_numeric(raw[column], errors="coerce")

date_month = raw["CloseDate"].dt.to_period("M").astype("string")
month_mismatches = date_month.notna() & date_month.ne(raw["SourceMonth"])
if month_mismatches.any():
    raise ValueError(
        f"Found {month_mismatches.sum():,} rows whose CloseDate month "
        "does not match the source-file month."
    )

source_summary = pd.DataFrame(source_rows)
source_summary.loc[:, "ca_sfr_positive_price_rows"] = [
    int(
        (
            raw.loc[raw["SourceFile"].eq(row["source_file"]), "StateOrProvince"].eq("CA")
            & raw.loc[raw["SourceFile"].eq(row["source_file"]), "PropertyType"].eq("Residential")
            & raw.loc[raw["SourceFile"].eq(row["source_file"]), "PropertySubType"].eq(
                "SingleFamilyResidence"
            )
            & raw.loc[raw["SourceFile"].eq(row["source_file"]), "ClosePrice"].gt(0)
        ).sum()
    )
    for row in source_rows
]
display(source_summary)
print(f"Imported {len(raw):,} raw rows from {len(source_summary)} monthly CSV files.")

,source_file,source_month,raw_rows,ca_sfr_positive_price_rows
0,CRMLSSold202506.csv,2025-06,22883,11700
1,CRMLSSold202507.csv,2025-07,23646,12113
2,CRMLSSold202508.csv,2025-08,22972,11452
3,CRMLSSold202509.csv,2025-09,22443,11456
4,CRMLSSold202510.csv,2025-10,23233,12028
5,CRMLSSold202511.csv,2025-11,19088,9739
6,CRMLSSold202512.csv,2025-12,20538,10455
7,CRMLSSold202601.csv,2026-01,16487,7490
8,CRMLSSold202602.csv,2026-02,18124,8550
9,CRMLSSold202603.csv,2026-03,22583,11177


Imported 283,176 raw rows from 13 monthly CSV files.


The source audit confirms a continuous, one-file-per-month
sequence and prevents accidental use of a future month. The `SourceMonth`
values are trace fields only; split assignment is chronological and June 2026
remains isolated from every tuning decision.

## Week 3 preprocessing on the shifted data

The Week 3 rules are applied directly to the new pandas import:

- retain California `Residential` + `SingleFamilyResidence` rows with a
  positive target;
- exclude identifiers, agent/office fields, exact addresses, listing-process
  leakage, and redundant/coarse fields by selecting the Week 3 retained inputs;
- convert invalid physical values to missing instead of dropping the property;
- clean ZIP/school labels and parse multi-value `Flooring`/`Levels` fields;
- retain missingness indicators;
- cap selected continuous predictors and remove target outliers using training
  data only, never the June holdout.

In [3]:
row_filter = (
    raw["StateOrProvince"].eq("CA")
    & raw["PropertyType"].eq("Residential")
    & raw["PropertySubType"].eq("SingleFamilyResidence")
    & raw["ClosePrice"].notna()
    & raw["ClosePrice"].gt(0)
)

df = raw.loc[row_filter].copy()
df["SaleMonth"] = df["CloseDate"].dt.to_period("M").astype("string")


def set_invalid_to_missing(frame, column, condition):
    before = int(frame[column].isna().sum())
    frame.loc[condition, column] = np.nan
    return int(frame[column].isna().sum()) - before


invalid_rule_counts = {}
invalid_rule_counts["Latitude"] = set_invalid_to_missing(
    df, "Latitude", ~df["Latitude"].between(32, 42)
)
invalid_rule_counts["Longitude"] = set_invalid_to_missing(
    df, "Longitude", ~df["Longitude"].between(-125, -114)
)
invalid_rule_counts["LotSizeSquareFeet"] = set_invalid_to_missing(
    df, "LotSizeSquareFeet", df["LotSizeSquareFeet"].le(100)
)
invalid_rule_counts["LivingArea"] = set_invalid_to_missing(
    df, "LivingArea", df["LivingArea"].le(0)
)
invalid_rule_counts["ParkingTotal"] = set_invalid_to_missing(
    df, "ParkingTotal", df["ParkingTotal"].lt(0)
)
invalid_rule_counts["GarageSpaces"] = set_invalid_to_missing(
    df, "GarageSpaces", df["GarageSpaces"].lt(0)
)
invalid_rule_counts["BathroomsTotalInteger"] = set_invalid_to_missing(
    df, "BathroomsTotalInteger", df["BathroomsTotalInteger"].le(0)
)
invalid_rule_counts["BedroomsTotal"] = set_invalid_to_missing(
    df, "BedroomsTotal", df["BedroomsTotal"].le(0)
)
invalid_rule_counts["MainLevelBedrooms_negative"] = set_invalid_to_missing(
    df, "MainLevelBedrooms", df["MainLevelBedrooms"].lt(0)
)
invalid_rule_counts["AssociationFee"] = set_invalid_to_missing(
    df, "AssociationFee", df["AssociationFee"].lt(0)
)
invalid_rule_counts["MainLevelBedrooms_consistency"] = set_invalid_to_missing(
    df,
    "MainLevelBedrooms",
    df["MainLevelBedrooms"].gt(df["BedroomsTotal"]),
)

count_upper_rules = {
    "ParkingTotal": 50,
    "GarageSpaces": 20,
    "BedroomsTotal": 20,
    "BathroomsTotalInteger": 20,
    "MainLevelBedrooms": 20,
}
for column, upper_bound in count_upper_rules.items():
    invalid_rule_counts[f"{column}_upper"] = set_invalid_to_missing(
        df, column, df[column].gt(upper_bound)
    )

categorical_inputs = [
    "Flooring",
    "Levels",
    "ViewYN",
    "PoolPrivateYN",
    "AttachedGarageYN",
    "FireplaceYN",
    "NewConstructionYN",
    "HighSchoolDistrict",
    "PostalCode",
]
for column in categorical_inputs:
    df[column] = df[column].astype("string").str.strip()
    df[column] = df[column].replace(r"^\s*$", pd.NA, regex=True)

df["PostalCode"] = df["PostalCode"].str.extract(r"(\d{5})", expand=False)
ambiguous_school_pattern = (
    r"(?i)^(other|see remarks|call listing office|"
    r"more than 1 district.*|.*inquire.*)$"
)
df["HighSchoolDistrict"] = df["HighSchoolDistrict"].replace(
    ambiguous_school_pattern,
    pd.NA,
    regex=True,
)

flooring_clean = (
    df["Flooring"]
    .str.lower()
    .str.replace(r"\s*,\s*", ",", regex=True)
)
df["Flooring_missing"] = df["Flooring"].isna().astype("int8")
flooring_materials = {
    "has_carpet": "carpet",
    "has_tile": "tile",
    "has_wood": "wood",
    "has_vinyl": "vinyl",
    "has_laminate": "laminate",
    "has_stone": "stone",
    "has_concrete": "concrete",
    "has_bamboo": "bamboo",
    "has_brick": "brick",
}
for new_column, token in flooring_materials.items():
    df[new_column] = flooring_clean.str.contains(
        rf"(?:^|,){token}(?:,|$)",
        regex=True,
        na=False,
    ).astype("int8")
df["Flooring_see_remarks"] = flooring_clean.str.contains(
    r"(?:^|,)seeremarks(?:,|$)",
    regex=True,
    na=False,
).astype("int8")

levels_clean = (
    df["Levels"]
    .str.lower()
    .str.replace(r"\s*,\s*", ",", regex=True)
)
df["Levels_missing"] = df["Levels"].isna().astype("int8")
level_patterns = {
    "level_one": "one",
    "level_two": "two",
    "level_three_or_more": "threeormore",
    "level_multisplit": "multisplit",
}
for new_column, token in level_patterns.items():
    df[new_column] = levels_clean.str.contains(
        rf"(?:^|,){token}(?:,|$)",
        regex=True,
        na=False,
    ).astype("int8")

BASE_FEATURE_COLUMNS = [
    "ViewYN",
    "PoolPrivateYN",
    "Latitude",
    "Longitude",
    "LivingArea",
    "AttachedGarageYN",
    "ParkingTotal",
    "YearBuilt",
    "BathroomsTotalInteger",
    "BedroomsTotal",
    "FireplaceYN",
    "MainLevelBedrooms",
    "NewConstructionYN",
    "GarageSpaces",
    "HighSchoolDistrict",
    "PostalCode",
    "AssociationFee",
    "LotSizeSquareFeet",
    "Flooring_missing",
    *flooring_materials.keys(),
    "Flooring_see_remarks",
    "Levels_missing",
    *level_patterns.keys(),
]

df = df[
    [
        "SourceFile",
        "SourceMonth",
        "CloseDate",
        "SaleMonth",
        "ClosePrice",
        *BASE_FEATURE_COLUMNS,
    ]
].copy()
df["split"] = np.where(df["SaleMonth"].eq(TEST_MONTH), "test", "train")

training_mask_before_outliers = df["split"].eq("train")
missing_indicator_sources = [
    column
    for column in BASE_FEATURE_COLUMNS
    if df.loc[training_mask_before_outliers, column].isna().any()
]
for column in missing_indicator_sources:
    df[f"{column}_missing_ind"] = df[column].isna().astype("int8")

train_before_outliers = df.loc[training_mask_before_outliers].copy()
price_per_living_area = (
    train_before_outliers["ClosePrice"] / train_before_outliers["LivingArea"]
).replace([np.inf, -np.inf], np.nan)
low_price_threshold = train_before_outliers["ClosePrice"].quantile(0.001)
top_price_threshold = train_before_outliers["ClosePrice"].quantile(0.999)
top_price_per_area = price_per_living_area.loc[
    train_before_outliers["ClosePrice"].gt(top_price_threshold)
]
q1 = top_price_per_area.quantile(0.25)
q3 = top_price_per_area.quantile(0.75)
iqr = q3 - q1
price_per_area_cutoff = (
    q3 + 1.5 * iqr
    if pd.notna(iqr) and iqr > 0
    else np.inf
)
remove_train_outlier = (
    train_before_outliers["ClosePrice"].lt(low_price_threshold)
    | price_per_living_area.gt(price_per_area_cutoff)
)
removed_train_indices = train_before_outliers.index[remove_train_outlier]
df = df.drop(index=removed_train_indices).reset_index(drop=True)

train_mask = df["split"].eq("train")
cap_values = {}
for column in ["LivingArea", "LotSizeSquareFeet", "AssociationFee"]:
    cap_values[column] = df.loc[train_mask, column].quantile(0.999)
    df[column] = df[column].clip(upper=cap_values[column])

preprocessing_summary = pd.DataFrame(
    [
        {"item": "raw imported rows", "value": len(raw)},
        {"item": "CA SFR rows with positive price", "value": int(row_filter.sum())},
        {"item": "training target outliers removed", "value": len(removed_train_indices)},
        {"item": "June test rows removed", "value": 0},
        {"item": "missing-indicator source columns", "value": len(missing_indicator_sources)},
        {"item": "W3 retained/derived base features", "value": len(BASE_FEATURE_COLUMNS)},
        {"item": "low target threshold (train only)", "value": low_price_threshold},
        {"item": "price/sqft upper cutoff (train only)", "value": price_per_area_cutoff},
    ]
)
display(preprocessing_summary)
display(
    pd.DataFrame(
        {
            "rule": invalid_rule_counts.keys(),
            "values_converted_to_missing": invalid_rule_counts.values(),
        }
    )
)
display(
    pd.DataFrame(
        {
            "continuous_feature": cap_values.keys(),
            "training_q999_cap": cap_values.values(),
        }
    )
)

,item,value
0,raw imported rows,"283,176.0000"
1,CA SFR rows with positive price,"143,071.0000"
2,training target outliers removed,154.0000
3,June test rows removed,0.0000
4,missing-indicator source columns,18.0000
5,W3 retained/derived base features,34.0000
6,low target threshold (train only),"90,409.7500"
7,price/sqft upper cutoff (train only),"11,150.1819"


,rule,values_converted_to_missing
0,Latitude,27
1,Longitude,39
2,LotSizeSquareFeet,377
3,LivingArea,53
4,ParkingTotal,22
5,GarageSpaces,0
6,BathroomsTotalInteger,61
7,BedroomsTotal,77
8,MainLevelBedrooms_negative,0
9,AssociationFee,0


,continuous_feature,training_q999_cap
0,LivingArea,"10,265.2200"
1,LotSizeSquareFeet,"5,593,624.8728"
2,AssociationFee,"4,054.8902"


**Interpretation.** Invalid predictor values are retained as missing so the
model does not lose otherwise usable sales. Target trimming and predictor caps
are learned only from the June 2025–May 2026 training set. Every valid June
2026 CA single-family sale remains in the final test set, which avoids an
artificially easy headline metric.

## Week 6 feature engineering on the shifted data

The Week 6 features are recreated from the shifted W3 table: bed/bath ratio,
age at sale, cyclical month, log living/lot area, and an authoritative CDE
2025–26 unified-school-district point-in-polygon feature. None uses `ClosePrice`
or a sale-process field.

In [4]:
bedrooms = pd.to_numeric(df["BedroomsTotal"], errors="coerce")
bathrooms = pd.to_numeric(df["BathroomsTotalInteger"], errors="coerce")
df["BedBathRatio"] = np.divide(
    bedrooms,
    bathrooms,
    out=np.full(len(df), np.nan, dtype=float),
    where=bathrooms.gt(0).to_numpy(),
)
df["BedBathRatio"] = df["BedBathRatio"].clip(lower=0, upper=10)

raw_property_age = (
    df["CloseDate"].dt.year
    - pd.to_numeric(df["YearBuilt"], errors="coerce")
)
df["PropertyAgeAtSale"] = raw_property_age.where(
    raw_property_age.between(0, 200)
)
df["PropertyAgeAtSale_missing_ind"] = (
    df["PropertyAgeAtSale"].isna().astype("int8")
)

sale_month_number = df["CloseDate"].dt.month
df["SaleMonthSin"] = np.sin(2 * np.pi * sale_month_number / 12)
df["SaleMonthCos"] = np.cos(2 * np.pi * sale_month_number / 12)
df["LogLivingArea"] = np.log1p(
    pd.to_numeric(df["LivingArea"], errors="coerce").clip(lower=0)
)
df["LogLotSizeSquareFeet"] = np.log1p(
    pd.to_numeric(df["LotSizeSquareFeet"], errors="coerce").clip(lower=0)
)

if not SCHOOL_DISTRICT_FILE.exists():
    raise FileNotFoundError(
        "Week 6 CDE boundary resource was not found: "
        f"{SCHOOL_DISTRICT_FILE}"
    )

districts = gpd.read_file(
    SCHOOL_DISTRICT_FILE,
    columns=["DistrictName", "DistrictType"],
).to_crs("EPSG:4326")
unified_districts = districts.loc[
    districts["DistrictType"].eq("Unified")
    & districts["DistrictName"].notna(),
    ["DistrictName", "geometry"],
].copy()
if unified_districts.empty:
    raise ValueError("No DistrictType='Unified' polygons found.")

points = gpd.GeoDataFrame(
    {"source_row_index": df.index},
    geometry=gpd.points_from_xy(df["Longitude"], df["Latitude"]),
    crs="EPSG:4326",
)
point_district_matches = gpd.sjoin(
    points,
    unified_districts,
    how="left",
    predicate="within",
)
match_counts = (
    point_district_matches.groupby("source_row_index")["DistrictName"]
    .count()
    .reindex(df.index, fill_value=0)
)
if match_counts.gt(1).any():
    raise ValueError("A property matched more than one Unified district polygon.")

district_name_by_row = (
    point_district_matches.groupby("source_row_index")["DistrictName"]
    .first()
    .reindex(df.index)
)
df["DistrictName"] = district_name_by_row
df["SchoolDistrictBoundaryMatched"] = (
    df["DistrictName"].notna().astype("int8")
)
df["DistrictName"] = df["DistrictName"].fillna("No boundary match")

feature_engineering_summary = pd.DataFrame(
    [
        {"item": "CDE polygons loaded", "value": len(districts)},
        {"item": "Unified polygons used", "value": len(unified_districts)},
        {
            "item": "properties matched to Unified district",
            "value": int(df["SchoolDistrictBoundaryMatched"].sum()),
        },
        {
            "item": "properties with no Unified match",
            "value": int((1 - df["SchoolDistrictBoundaryMatched"]).sum()),
        },
        {
            "item": "unique matched districts",
            "value": int(
                df.loc[
                    df["SchoolDistrictBoundaryMatched"].eq(1),
                    "DistrictName",
                ].nunique()
            ),
        },
        {
            "item": "new Week 6 engineered features",
            "value": 9,
        },
    ]
)
display(feature_engineering_summary)

,item,value
0,CDE polygons loaded,936
1,Unified polygons used,345
2,properties matched to Unified district,108348
3,properties with no Unified match,34569
4,unique matched districts,322
5,new Week 6 engineered features,9


**Interpretation.** These features represent intrinsic utility, aging,
seasonal valuation timing, diminishing size effects, and target-independent
location. The district join is a demand/location proxy, not a causal claim.
Properties outside unified-only polygon coverage are retained with an explicit
no-match category and flag.

## Final split and leakage audit

The final holdout is every eligible June 2026 row. `CloseDate`, `SaleMonth`,
source-file fields, and `ClosePrice` are trace/target fields, not predictors.
Imputation, scaling, and one-hot encoding are fitted inside every
chronological validation fold and then refitted on the full training window.

In [5]:
TRACE_COLUMNS = [
    "split",
    "SourceFile",
    "SourceMonth",
    "CloseDate",
    "SaleMonth",
    "ClosePrice",
]
train = df.loc[df["split"].eq("train")].copy()
test = df.loc[df["split"].eq("test")].copy()

expected_train_months = pd.period_range(
    TRAIN_START_MONTH,
    TRAIN_END_MONTH,
    freq="M",
).astype(str).tolist()
actual_train_months = sorted(train["SaleMonth"].dropna().unique().tolist())
actual_test_months = sorted(test["SaleMonth"].dropna().unique().tolist())
if actual_train_months != expected_train_months:
    raise ValueError(
        f"Expected training months {expected_train_months}, "
        f"found {actual_train_months}"
    )
if actual_test_months != [TEST_MONTH]:
    raise ValueError(
        f"Expected test month {[TEST_MONTH]}, found {actual_test_months}"
    )

feature_columns = [
    column
    for column in df.columns
    if column not in TRACE_COLUMNS
]
leakage_matches = [
    column
    for column in feature_columns
    if any(term.lower() in column.lower() for term in FORBIDDEN_FEATURE_TERMS)
]
if leakage_matches:
    raise ValueError(f"Leakage columns found: {leakage_matches}")

X_train = train[feature_columns].copy()
X_test = test[feature_columns].copy()
y_train = train["ClosePrice"].to_numpy(dtype=np.float64)
y_test = test["ClosePrice"].to_numpy(dtype=np.float64)

numeric_columns = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_columns = [
    column for column in feature_columns if column not in numeric_columns
]
for column in categorical_columns:
    X_train[column] = X_train[column].astype(object)
    X_test[column] = X_test[column].astype(object)
X_train = X_train.where(pd.notna(X_train), np.nan)
X_test = X_test.where(pd.notna(X_test), np.nan)

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ("scaler", StandardScaler()),
    ]
)
categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="constant", fill_value="Unknown"),
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="infrequent_if_exist",
                min_frequency=50,
                dtype=np.float32,
            ),
        ),
    ]
)
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_columns),
        ("cat", categorical_transformer, categorical_columns),
    ],
    sparse_threshold=0.3,
)

split_summary = pd.DataFrame(
    [
        {
            "split": "training",
            "months": f"{TRAIN_START_MONTH} to {TRAIN_END_MONTH}",
            "rows": len(train),
        },
        {
            "split": "untouched test",
            "months": TEST_MONTH,
            "rows": len(test),
        },
    ]
)
display(split_summary)
display(
    pd.DataFrame(
        [
            {"audit": "readable predictor columns", "value": len(feature_columns)},
            {"audit": "numeric predictor columns", "value": len(numeric_columns)},
            {
                "audit": "categorical predictor columns",
                "value": len(categorical_columns),
            },
            {"audit": "forbidden feature matches", "value": len(leakage_matches)},
            {"audit": "test rows removed", "value": 0},
        ]
    )
)

,split,months,rows
0,training,2025-06 to 2026-05,130060
1,untouched test,2026-06,12857


,audit,value
0,readable predictor columns,61
1,numeric predictor columns,53
2,categorical predictor columns,8
3,forbidden feature matches,0
4,test rows removed,0


**Interpretation.** The split reproduces production timing: the model learns
from information available through May 2026 and estimates homes sold in June.
The untouched test set is larger than the prior May holdout and no test
observation is removed for being difficult or expensive.

## Metrics, chronological folds, and light tuning grid

Three rolling-origin folds validate on March, April, and May 2026. Model
selection uses mean validation **MdAPE** first because a typical percentage
error is meaningful across California price levels; mean validation **R²** is
the tie-breaker. MAE, RMSE, MAPE, and fold dispersion are also reported.

Each algorithm gets three deliberately small combinations of tree depth,
learning rate, and estimator count.

In [6]:
def regression_metrics(y_true, y_pred):
    absolute_percentage_error = np.abs(y_true - y_pred) / y_true
    return {
        "r2": float(r2_score(y_true, y_pred)),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "median_absolute_error": float(
            median_absolute_error(y_true, y_pred)
        ),
        "mape": float(np.mean(absolute_percentage_error)),
        "mdape": float(np.median(absolute_percentage_error)),
    }


def build_estimator(family, params):
    if family == "XGBoost":
        return xgb.XGBRegressor(
            objective="reg:squarederror",
            eval_metric="rmse",
            tree_method="hist",
            subsample=0.85,
            colsample_bytree=0.80,
            reg_lambda=5.0,
            random_state=RANDOM_STATE,
            n_jobs=N_JOBS,
            verbosity=0,
            **params,
        )
    if family == "LightGBM":
        return lgb.LGBMRegressor(
            objective="regression",
            subsample=0.85,
            subsample_freq=1,
            colsample_bytree=0.80,
            reg_lambda=5.0,
            min_child_samples=30,
            random_state=RANDOM_STATE,
            n_jobs=N_JOBS,
            verbosity=-1,
            **params,
        )
    raise ValueError(f"Unknown gradient boosting family: {family}")


candidate_specs = [
    {
        "family": "XGBoost",
        "model": "XGBoost (depth 6, lr 0.05, 300 trees)",
        "params": {
            "max_depth": 6,
            "learning_rate": 0.05,
            "n_estimators": 300,
        },
    },
    {
        "family": "XGBoost",
        "model": "XGBoost (depth 8, lr 0.05, 500 trees)",
        "params": {
            "max_depth": 8,
            "learning_rate": 0.05,
            "n_estimators": 500,
        },
    },
    {
        "family": "XGBoost",
        "model": "XGBoost (depth 6, lr 0.03, 700 trees)",
        "params": {
            "max_depth": 6,
            "learning_rate": 0.03,
            "n_estimators": 700,
        },
    },
    {
        "family": "LightGBM",
        "model": "LightGBM (depth 6, lr 0.05, 300 trees)",
        "params": {
            "max_depth": 6,
            "num_leaves": 31,
            "learning_rate": 0.05,
            "n_estimators": 300,
        },
    },
    {
        "family": "LightGBM",
        "model": "LightGBM (depth 10, lr 0.05, 500 trees)",
        "params": {
            "max_depth": 10,
            "num_leaves": 63,
            "learning_rate": 0.05,
            "n_estimators": 500,
        },
    },
    {
        "family": "LightGBM",
        "model": "LightGBM (depth 10, lr 0.03, 700 trees)",
        "params": {
            "max_depth": 10,
            "num_leaves": 63,
            "learning_rate": 0.03,
            "n_estimators": 700,
        },
    },
]

cv_folds = []
cv_fold_rows = []
for fold_number, validation_month in enumerate(
    CV_VALIDATION_MONTHS,
    start=1,
):
    fold_train_indices = np.flatnonzero(
        train["SaleMonth"].lt(validation_month).to_numpy()
    )
    fold_validation_indices = np.flatnonzero(
        train["SaleMonth"].eq(validation_month).to_numpy()
    )
    if len(fold_train_indices) == 0 or len(fold_validation_indices) == 0:
        raise ValueError(f"Empty chronological fold for {validation_month}")
    cv_folds.append((fold_train_indices, fold_validation_indices))
    cv_fold_rows.append(
        {
            "fold": fold_number,
            "train_months": (
                f"{train.iloc[fold_train_indices]['SaleMonth'].min()} to "
                f"{train.iloc[fold_train_indices]['SaleMonth'].max()}"
            ),
            "validation_month": validation_month,
            "train_rows": len(fold_train_indices),
            "validation_rows": len(fold_validation_indices),
            "shuffle": False,
        }
    )

cv_fold_plan = pd.DataFrame(cv_fold_rows)
cv_fold_plan.to_csv(
    OUTPUT_DIR / "advanced_model_cv_fold_plan.csv",
    index=False,
)
display(cv_fold_plan)
display(
    pd.DataFrame(candidate_specs)[["family", "model", "params"]]
)

,fold,train_months,validation_month,train_rows,validation_rows,shuffle
0,1,2025-06 to 2026-02,2026-03,94867,11166,False
1,2,2025-06 to 2026-03,2026-04,106033,12020,False
2,3,2025-06 to 2026-04,2026-05,118053,12007,False


,family,model,params
0,XGBoost,"XGBoost (depth 6, lr 0.05, 300 trees)","{'max_depth': 6, 'learning_rate': 0.05, 'n_estimators': 300}"
1,XGBoost,"XGBoost (depth 8, lr 0.05, 500 trees)","{'max_depth': 8, 'learning_rate': 0.05, 'n_estimators': 500}"
2,XGBoost,"XGBoost (depth 6, lr 0.03, 700 trees)","{'max_depth': 6, 'learning_rate': 0.03, 'n_estimators': 700}"
3,LightGBM,"LightGBM (depth 6, lr 0.05, 300 trees)","{'max_depth': 6, 'num_leaves': 31, 'learning_rate': 0.05, 'n_estimators': 300}"
4,LightGBM,"LightGBM (depth 10, lr 0.05, 500 trees)","{'max_depth': 10, 'num_leaves': 63, 'learning_rate': 0.05, 'n_estimators': 500}"
5,LightGBM,"LightGBM (depth 10, lr 0.03, 700 trees)","{'max_depth': 10, 'num_leaves': 63, 'learning_rate': 0.03, 'n_estimators': 700}"


In [7]:
cv_detail_rows = []
cv_start = time.time()

for fold_number, (fold_train_indices, fold_validation_indices) in enumerate(
    cv_folds,
    start=1,
):
    fold_preprocessor = clone(preprocessor)
    fold_X_train = fold_preprocessor.fit_transform(
        X_train.iloc[fold_train_indices]
    )
    fold_X_validation = fold_preprocessor.transform(
        X_train.iloc[fold_validation_indices]
    )
    fold_y_train = y_train[fold_train_indices]
    fold_y_validation = y_train[fold_validation_indices]

    for spec in candidate_specs:
        fit_start = time.time()
        estimator = build_estimator(spec["family"], spec["params"])
        estimator.fit(fold_X_train, fold_y_train)
        validation_predictions = estimator.predict(fold_X_validation)
        metrics = regression_metrics(
            fold_y_validation,
            validation_predictions,
        )
        cv_detail_rows.append(
            {
                "fold": fold_number,
                "validation_month": CV_VALIDATION_MONTHS[fold_number - 1],
                "family": spec["family"],
                "model": spec["model"],
                "train_rows": len(fold_train_indices),
                "validation_rows": len(fold_validation_indices),
                "encoded_feature_count": fold_X_train.shape[1],
                "validation_r2": metrics["r2"],
                "validation_mae": metrics["mae"],
                "validation_rmse": metrics["rmse"],
                "validation_mape": metrics["mape"],
                "validation_mdape": metrics["mdape"],
                "fit_seconds": time.time() - fit_start,
                "params": json.dumps(spec["params"], sort_keys=True),
            }
        )

    del fold_preprocessor, fold_X_train, fold_X_validation
    gc.collect()
    print(
        f"Completed fold {fold_number}/{len(cv_folds)} "
        f"({CV_VALIDATION_MONTHS[fold_number - 1]})."
    )

cv_detail = pd.DataFrame(cv_detail_rows)
cv_detail.to_csv(
    OUTPUT_DIR / "advanced_model_cv_detail_results.csv",
    index=False,
)

validation_results = (
    cv_detail.groupby(["family", "model", "params"], as_index=False)
    .agg(
        cv_folds=("fold", "nunique"),
        mean_validation_r2=("validation_r2", "mean"),
        std_validation_r2=("validation_r2", "std"),
        mean_validation_mae=("validation_mae", "mean"),
        mean_validation_rmse=("validation_rmse", "mean"),
        mean_validation_mape=("validation_mape", "mean"),
        mean_validation_mdape=("validation_mdape", "mean"),
        std_validation_mdape=("validation_mdape", "std"),
        total_fit_seconds=("fit_seconds", "sum"),
    )
)
validation_results["selected"] = False

selected_rows = []
for family in ["XGBoost", "LightGBM"]:
    selected = (
        validation_results.loc[validation_results["family"].eq(family)]
        .sort_values(
            ["mean_validation_mdape", "mean_validation_r2"],
            ascending=[True, False],
        )
        .iloc[0]
    )
    selected_rows.append(selected)
    validation_results.loc[
        validation_results["model"].eq(selected["model"]),
        "selected",
    ] = True

validation_results.to_csv(
    OUTPUT_DIR / "advanced_model_validation_results.csv",
    index=False,
)
selected_models = pd.DataFrame(selected_rows)

display(
    validation_results.drop(columns="params").sort_values(
        ["family", "mean_validation_mdape"]
    )
)
print(
    "Selected models:",
    ", ".join(selected_models["model"].tolist()),
)
print(f"Chronological tuning seconds: {time.time() - cv_start:,.1f}")

Completed fold 1/3 (2026-03).


Completed fold 2/3 (2026-04).


Completed fold 3/3 (2026-05).


,family,model,cv_folds,mean_validation_r2,std_validation_r2,mean_validation_mae,mean_validation_rmse,mean_validation_mape,mean_validation_mdape,std_validation_mdape,total_fit_seconds,selected
1,LightGBM,"LightGBM (depth 10, lr 0.05, 500 trees)",3,0.7999,0.0346,"237,208.9815","660,094.9962",0.1796,0.1229,0.0017,10.2643,True
0,LightGBM,"LightGBM (depth 10, lr 0.03, 700 trees)",3,0.7982,0.0324,"239,342.8533","663,117.0834",0.1821,0.1231,0.0011,14.5753,False
2,LightGBM,"LightGBM (depth 6, lr 0.05, 300 trees)",3,0.7696,0.0412,"281,138.6902","708,642.8284",0.2260,0.1556,0.0008,4.4919,False
5,XGBoost,"XGBoost (depth 8, lr 0.05, 500 trees)",3,0.8026,0.0331,"230,587.2148","655,463.7816",0.1720,0.1182,0.0030,7.1849,True
3,XGBoost,"XGBoost (depth 6, lr 0.03, 700 trees)",3,0.7954,0.0277,"258,189.1201","667,599.6087",0.2026,0.1407,0.0019,7.1217,False
4,XGBoost,"XGBoost (depth 6, lr 0.05, 300 trees)",3,0.7826,0.0317,"272,105.1478","688,308.2109",0.2168,0.1500,0.0029,3.4361,False


Selected models: XGBoost (depth 8, lr 0.05, 500 trees), LightGBM (depth 10, lr 0.05, 500 trees)
Chronological tuning seconds: 49.9


**Tuning interpretation.** The selected XGBoost configuration (depth 8,
learning rate 0.05, 500 trees) produced mean validation R² of **0.803** and
mean MdAPE of **11.82%**. The selected LightGBM configuration (depth 10,
learning rate 0.05, 500 trees) was close at **0.800 R²** and **12.29% MdAPE**.
XGBoost's MdAPE standard deviation was only 0.30 percentage points across the
three validation months, so its gain did not come from one unusually easy
fold. The smaller 300-tree candidates underfit; increasing to 700 trees at the
lower learning rate did not improve typical error enough to justify the extra
complexity.

## Training-window diagnostic (training data only)

The assignment fixes the final shifted window at twelve months. As a
sensitivity check, the stronger selected booster is also trained on the latest
3, 6, 9, and 11 months available before May 2026 and validated on May. The
eleven-month row is the closest leakage-free proxy for the requested
twelve-month final fit because May itself cannot be both training and
validation data.

In [8]:
overall_selected = (
    selected_models.sort_values(
        ["mean_validation_mdape", "mean_validation_r2"],
        ascending=[True, False],
    )
    .iloc[0]
)
overall_selected_family = overall_selected["family"]
overall_selected_model = overall_selected["model"]
overall_selected_params = json.loads(overall_selected["params"])

window_validation_month = TRAIN_END_MONTH
months_before_window_validation = sorted(
    train.loc[train["SaleMonth"].lt(window_validation_month), "SaleMonth"]
    .dropna()
    .unique()
    .tolist()
)
window_rows = []

for window_months in [3, 6, 9, 11]:
    selected_train_months = months_before_window_validation[-window_months:]
    window_train_indices = np.flatnonzero(
        train["SaleMonth"].isin(selected_train_months).to_numpy()
    )
    window_validation_indices = np.flatnonzero(
        train["SaleMonth"].eq(window_validation_month).to_numpy()
    )

    window_preprocessor = clone(preprocessor)
    window_X_train = window_preprocessor.fit_transform(
        X_train.iloc[window_train_indices]
    )
    window_X_validation = window_preprocessor.transform(
        X_train.iloc[window_validation_indices]
    )
    window_estimator = build_estimator(
        overall_selected_family,
        overall_selected_params,
    )
    window_estimator.fit(
        window_X_train,
        y_train[window_train_indices],
    )
    window_predictions = window_estimator.predict(window_X_validation)
    window_metrics = regression_metrics(
        y_train[window_validation_indices],
        window_predictions,
    )
    window_rows.append(
        {
            "model": overall_selected_model,
            "training_window_months": window_months,
            "train_months": (
                f"{selected_train_months[0]} to "
                f"{selected_train_months[-1]}"
            ),
            "validation_month": window_validation_month,
            "train_rows": len(window_train_indices),
            "validation_rows": len(window_validation_indices),
            "validation_r2": window_metrics["r2"],
            "validation_mae": window_metrics["mae"],
            "validation_rmse": window_metrics["rmse"],
            "validation_mape": window_metrics["mape"],
            "validation_mdape": window_metrics["mdape"],
        }
    )
    del (
        window_preprocessor,
        window_X_train,
        window_X_validation,
        window_estimator,
    )
    gc.collect()

window_validation = pd.DataFrame(window_rows)
window_validation.to_csv(
    OUTPUT_DIR / "training_window_validation.csv",
    index=False,
)
display(window_validation)

,model,training_window_months,train_months,validation_month,train_rows,validation_rows,validation_r2,validation_mae,validation_rmse,validation_mape,validation_mdape
0,"XGBoost (depth 8, lr 0.05, 500 trees)",3,2026-02 to 2026-04,2026-05,31720,12007,0.7785,"230,082.5157","640,565.1315",0.1674,0.1104
1,"XGBoost (depth 8, lr 0.05, 500 trees)",6,2025-11 to 2026-04,2026-05,59365,12007,0.7915,"226,303.3041","621,517.0693",0.1661,0.1135
2,"XGBoost (depth 8, lr 0.05, 500 trees)",9,2025-08 to 2026-04,2026-05,94266,12007,0.8096,"222,698.0597","593,829.2205",0.1646,0.1154
3,"XGBoost (depth 8, lr 0.05, 500 trees)",11,2025-06 to 2026-04,2026-05,118053,12007,0.8140,"229,049.8712","587,016.9991",0.1730,0.1208


**Window interpretation.** No proxy window dominated every metric. Three
months had the lowest May MdAPE (**11.04%**), nine months had the lowest MAE
(**$222,698**), and eleven months had the best R² (**0.814**) and RMSE
(**$587,017**). The final model therefore uses the assignment's required June
2025–May 2026 twelve-month window: it extends the strongest variance/tail-risk
proxy to a complete seasonal cycle and the largest recent sample without
admitting any June outcome. Window length should be retested across multiple
future rolling holdouts rather than selected from one May result alone.

## Final June 2026 test metrics

After all choices are frozen, the preprocessing pipeline is fitted once on
June 2025–May 2026. The selected XGBoost and LightGBM configurations are
evaluated on the same June 2026 rows. The Week 6 Random Forest setting is
retrained on the same matrix as a fair, shifted-month reference.

In [9]:
final_preprocessor = clone(preprocessor)
preprocess_start = time.time()
X_train_processed = final_preprocessor.fit_transform(X_train)
X_test_processed = final_preprocessor.transform(X_test)
preprocess_seconds = time.time() - preprocess_start
encoded_feature_names = final_preprocessor.get_feature_names_out()

final_specs = []
for _, selected in selected_models.iterrows():
    family = selected["family"]
    params = json.loads(selected["params"])
    final_specs.append(
        {
            "family": family,
            "model": selected["model"],
            "estimator": build_estimator(family, params),
            "params": params,
            "role": "tuned gradient booster",
        }
    )

random_forest_params = {
    "n_estimators": 60,
    "max_depth": 30,
    "min_samples_leaf": 10,
    "max_features": 0.7,
    "random_state": RANDOM_STATE,
    "n_jobs": N_JOBS,
}
final_specs.append(
    {
        "family": "Random Forest",
        "model": "Week 6 Random Forest reference",
        "estimator": RandomForestRegressor(**random_forest_params),
        "params": random_forest_params,
        "role": "same-month historical-family reference",
    }
)

test_metric_rows = []
fitted_models = []
predictions = test[
    ["SourceFile", "CloseDate", "SaleMonth", "ClosePrice"]
].reset_index(drop=True)

for spec in final_specs:
    fit_start = time.time()
    estimator = spec["estimator"]
    estimator.fit(X_train_processed, y_train)
    fit_seconds = time.time() - fit_start
    train_predictions = estimator.predict(X_train_processed)
    test_predictions = estimator.predict(X_test_processed)
    train_metrics = regression_metrics(y_train, train_predictions)
    test_metrics = regression_metrics(y_test, test_predictions)

    prediction_prefix = (
        spec["family"].lower().replace(" ", "_")
    )
    prediction_column = f"{prediction_prefix}_prediction"
    predictions[prediction_column] = test_predictions
    predictions[
        f"{prediction_prefix}_absolute_error"
    ] = np.abs(y_test - test_predictions)
    predictions[
        f"{prediction_prefix}_absolute_percentage_error"
    ] = np.abs(y_test - test_predictions) / y_test

    test_metric_rows.append(
        {
            "model": spec["model"],
            "family": spec["family"],
            "role": spec["role"],
            "train_rows": len(train),
            "test_rows": len(test),
            "encoded_feature_count": X_train_processed.shape[1],
            "train_months": f"{TRAIN_START_MONTH} to {TRAIN_END_MONTH}",
            "test_month": TEST_MONTH,
            "train_r2": train_metrics["r2"],
            "test_r2": test_metrics["r2"],
            "test_mae": test_metrics["mae"],
            "test_rmse": test_metrics["rmse"],
            "test_median_absolute_error": test_metrics[
                "median_absolute_error"
            ],
            "test_mape": test_metrics["mape"],
            "test_mdape": test_metrics["mdape"],
            "fit_seconds": fit_seconds,
            "selected_parameters": json.dumps(
                spec["params"],
                sort_keys=True,
            ),
        }
    )
    fitted_models.append((spec, estimator))

test_metrics = pd.DataFrame(test_metric_rows).sort_values(
    ["test_mdape", "test_r2"],
    ascending=[True, False],
)
test_metrics.to_csv(
    OUTPUT_DIR / "advanced_model_test_metrics.csv",
    index=False,
)
predictions.to_csv(
    OUTPUT_DIR / "advanced_model_test_predictions.csv.gz",
    index=False,
    compression="gzip",
)

test_metric_display = test_metrics[
    [
        "model",
        "role",
        "train_r2",
        "test_r2",
        "test_mae",
        "test_rmse",
        "test_mape",
        "test_mdape",
        "fit_seconds",
    ]
].copy()
display(test_metric_display)
print(
    f"Final encoded features: {X_train_processed.shape[1]:,}; "
    f"preprocessing seconds: {preprocess_seconds:,.1f}"
)

,model,role,train_r2,test_r2,test_mae,test_rmse,test_mape,test_mdape,fit_seconds
2,Week 6 Random Forest reference,same-month historical-family reference,0.8076,0.7486,"232,645.9745","770,384.5362",0.2369,0.0914,114.8976
0,"XGBoost (depth 8, lr 0.05, 500 trees)",tuned gradient booster,0.9349,0.8116,"232,373.3406","666,902.5662",0.2520,0.1177,2.6245
1,"LightGBM (depth 10, lr 0.05, 500 trees)",tuned gradient booster,0.8655,0.7975,"242,444.3286","691,360.4275",0.2656,0.1223,3.7775


Final encoded features: 1,193; preprocessing seconds: 1.0


**Test interpretation.** XGBoost leads the June holdout on R² (**0.812**),
RMSE (**$666,903**), and narrowly on MAE (**$232,373**). LightGBM is second on
R² (**0.798**) and RMSE (**$691,360**). The Random Forest reference has lower
relative error—**23.69% MAPE** and **9.14% MdAPE**, versus XGBoost's **25.20%**
and **11.77%**—so no model dominates every measure. XGBoost is the best overall
statewide candidate when explaining market-wide price variance and limiting
large dollar misses matter; Random Forest remains a valuable benchmark, or
possible ensemble component, for the typical home's percentage error.

XGBoost's June R² and MdAPE are also very close to its chronological validation
averages (0.803 and 11.82%), which is encouraging evidence of temporal
generalization. It is evidence, not a guarantee of future-month accuracy.

## Error diagnostics and feature signals

Quintile-level metrics reveal whether a good statewide average hides weakness
in a price segment. Feature importance is directional only in the sense of
ranking usage/gain; it does not establish causality.

In [10]:
price_band_rows = []
band_frame = predictions.copy()
band_frame["price_band"] = pd.qcut(
    band_frame["ClosePrice"],
    q=5,
    duplicates="drop",
)

for spec, _ in fitted_models:
    prefix = spec["family"].lower().replace(" ", "_")
    prediction_column = f"{prefix}_prediction"
    absolute_error_column = f"{prefix}_absolute_error"
    ape_column = f"{prefix}_absolute_percentage_error"
    for price_band, group in band_frame.groupby(
        "price_band",
        observed=True,
    ):
        price_band_rows.append(
            {
                "model": spec["model"],
                "price_band": str(price_band),
                "rows": len(group),
                "mean_close_price": group["ClosePrice"].mean(),
                "r2_within_band": r2_score(
                    group["ClosePrice"],
                    group[prediction_column],
                ),
                "mae": group[absolute_error_column].mean(),
                "mape": group[ape_column].mean(),
                "mdape": group[ape_column].median(),
            }
        )

price_band_metrics = pd.DataFrame(price_band_rows)
price_band_metrics.to_csv(
    OUTPUT_DIR / "advanced_model_price_band_metrics.csv",
    index=False,
)

importance_rows = []
for spec, estimator in fitted_models:
    values = np.asarray(estimator.feature_importances_)
    ranking = np.argsort(values)[::-1][:25]
    for rank, feature_index in enumerate(ranking, start=1):
        importance_rows.append(
            {
                "model": spec["model"],
                "rank": rank,
                "feature": encoded_feature_names[feature_index],
                "importance": float(values[feature_index]),
            }
        )

feature_importance = pd.DataFrame(importance_rows)
feature_importance.to_csv(
    OUTPUT_DIR / "advanced_model_feature_importance.csv",
    index=False,
)

display(price_band_metrics)
display(feature_importance.groupby("model").head(10))

,model,price_band,rows,mean_close_price,r2_within_band,mae,mape,mdape
0,"XGBoost (depth 8, lr 0.05, 500 trees)","(579.999, 575000.0]",2616,"432,194.9344",-1.3135,"101,245.1231",0.6497,0.1506
1,"XGBoost (depth 8, lr 0.05, 500 trees)","(575000.0, 800000.0]",2546,"693,248.3790",-4.2946,"99,671.2659",0.1452,0.1010
2,"XGBoost (depth 8, lr 0.05, 500 trees)","(800000.0, 1100000.0]",2561,"935,328.3944",-10.2184,"132,734.7263",0.1414,0.0988
3,"XGBoost (depth 8, lr 0.05, 500 trees)","(1100000.0, 1665000.0]",2563,"1,357,183.2723",-1.7517,"190,314.6648",0.1398,0.1084
4,"XGBoost (depth 8, lr 0.05, 500 trees)","(1665000.0, 46950000.0]",2571,"3,120,120.4028",0.7178,"638,387.2563",0.1751,0.1393
5,"LightGBM (depth 10, lr 0.05, 500 trees)","(579.999, 575000.0]",2616,"432,194.9344",-1.7538,"105,817.0655",0.6890,0.1498
6,"LightGBM (depth 10, lr 0.05, 500 trees)","(575000.0, 800000.0]",2546,"693,248.3790",-4.8884,"103,830.5358",0.1514,0.1048
7,"LightGBM (depth 10, lr 0.05, 500 trees)","(800000.0, 1100000.0]",2561,"935,328.3944",-7.5046,"137,972.2203",0.1472,0.1034
8,"LightGBM (depth 10, lr 0.05, 500 trees)","(1100000.0, 1665000.0]",2563,"1,357,183.2723",-2.0497,"199,256.4660",0.1465,0.1119
9,"LightGBM (depth 10, lr 0.05, 500 trees)","(1665000.0, 46950000.0]",2571,"3,120,120.4028",0.6941,"665,848.1396",0.1845,0.1460


,model,rank,feature,importance
0,"XGBoost (depth 8, lr 0.05, 500 trees)",1,num__missingindicator_BedBathRatio,0.0487
1,"XGBoost (depth 8, lr 0.05, 500 trees)",2,num__missingindicator_BathroomsTotalInteger,0.0281
2,"XGBoost (depth 8, lr 0.05, 500 trees)",3,cat__DistrictName_Newport-Mesa Unified,0.0277
3,"XGBoost (depth 8, lr 0.05, 500 trees)",4,cat__HighSchoolDistrict_Newport Mesa Unified,0.0235
4,"XGBoost (depth 8, lr 0.05, 500 trees)",5,cat__PoolPrivateYN_Unknown,0.0224
5,"XGBoost (depth 8, lr 0.05, 500 trees)",6,num__LogLivingArea,0.0196
6,"XGBoost (depth 8, lr 0.05, 500 trees)",7,num__BathroomsTotalInteger_missing_ind,0.0195
7,"XGBoost (depth 8, lr 0.05, 500 trees)",8,num__AssociationFee_missing_ind,0.0190
8,"XGBoost (depth 8, lr 0.05, 500 trees)",9,num__missingindicator_GarageSpaces,0.0163
9,"XGBoost (depth 8, lr 0.05, 500 trees)",10,num__LivingArea_missing_ind,0.0143


**Diagnostic interpretation.** XGBoost's median percentage error ranges from
**9.88%** in the middle price quintile to **15.06%** in the lowest quintile;
its top-quintile MAE is **$638,387** and within-band R² is **0.718**. The very
low test minimum ($580) makes mean percentage error especially sensitive in
the first quintile, so MdAPE and dollar error are more representative there.
The Random Forest has lower MdAPE in every quintile, while XGBoost better
captures overall and high-end variance. Feature importance ranks predictive
use only; it does not mean that changing a school district or ZIP would
causally change value.

## Comparison with prior modeling weeks

Historical Week 4–6 metrics used May 2026 as their holdout and are retained for
lineage. They are **not** directly comparable to the June 2026 metrics because
market mix and price extremes change month to month. The retrained Week 6
Random Forest reference above is the fair same-data comparison.

In [11]:
historical_sources = [
    (
        "Week 4",
        PROJECT_ROOT / "W4 Baseline Model" / "baseline_model_results.csv",
    ),
    (
        "Week 5",
        PROJECT_ROOT / "W5 Additional Models" / "model_comparison_results.csv",
    ),
    (
        "Week 6",
        PROJECT_ROOT / "W6 Feature Engineering" / "model_comparison_results.csv",
    ),
]
history_frames = []
comparison_columns = [
    "model",
    "test_month",
    "test_r2",
    "test_mae",
    "test_rmse",
    "test_mape",
    "test_mdape",
]

for week, source_path in historical_sources:
    if source_path.exists():
        history = pd.read_csv(source_path)
        history = history[comparison_columns].copy()
        history.insert(0, "week", week)
        history["comparison_basis"] = "historical holdout; context only"
        history_frames.append(history)

w7_comparison = test_metrics[comparison_columns].copy()
w7_comparison.insert(0, "week", "Week 7")
w7_comparison["comparison_basis"] = "June 2026 shifted holdout"
all_weeks_comparison = pd.concat(
    [*history_frames, w7_comparison],
    ignore_index=True,
)
all_weeks_comparison.to_csv(
    OUTPUT_DIR / "all_weeks_model_comparison.csv",
    index=False,
)
display(all_weeks_comparison)

,week,model,test_month,test_r2,test_mae,test_rmse,test_mape,test_mdape,comparison_basis
0,Week 4,Linear Regression,2026-05,0.4714,"363,777.9113","1,220,014.7686",0.3777,0.2241,historical holdout; context only
1,Week 5,Linear Regression (baseline),2026-05,0.4714,"363,777.9113","1,220,014.7686",0.3777,0.2241,historical holdout; context only
2,Week 5,"Decision Tree (depth 24, leaf 10)",2026-05,0.4398,"275,506.3772","1,255,925.9252",0.1978,0.1064,historical holdout; context only
3,Week 5,"Random Forest (60 trees, depth 30, leaf 10, max_features 0.7)",2026-05,0.5139,"232,766.8978","1,170,010.9042",0.1824,0.0897,historical holdout; context only
4,Week 6,Linear Regression (baseline),2026-05,0.4945,"322,806.3089","1,193,077.0011",0.3034,0.1774,historical holdout; context only
5,Week 6,"Decision Tree (depth 24, leaf 10)",2026-05,0.4729,"276,724.0230","1,218,313.1155",0.1965,0.1072,historical holdout; context only
6,Week 6,"Random Forest (60 trees, depth 30, leaf 10, max_features 0.7)",2026-05,0.5216,"234,157.4583","1,160,644.1715",0.1794,0.0897,historical holdout; context only
7,Week 7,Week 6 Random Forest reference,2026-06,0.7486,"232,645.9745","770,384.5362",0.2369,0.0914,June 2026 shifted holdout
8,Week 7,"XGBoost (depth 8, lr 0.05, 500 trees)",2026-06,0.8116,"232,373.3406","666,902.5662",0.2520,0.1177,June 2026 shifted holdout
9,Week 7,"LightGBM (depth 10, lr 0.05, 500 trees)",2026-06,0.7975,"242,444.3286","691,360.4275",0.2656,0.1223,June 2026 shifted holdout


**Cross-week interpretation.** The recommended statewide production candidate
is the selected XGBoost model because, on the fair same-month June comparison,
it improves R² from the shifted Random Forest's **0.749** to **0.812**, lowers
RMSE from **$770,385** to **$666,903**, and is essentially tied on MAE. The
Random Forest's **9.14% MdAPE** remains better than XGBoost's **11.77%**, so it
should be retained as a benchmark and considered for an ensemble. Historical
Week 4–6 metrics establish lineage but use May 2026; they are not treated as
apples-to-apples evidence that one model is superior.

## Weekly report and deliverable files

The cells below write a short weekly report and a plain-language explanation
of every W7 file. The report distinguishes measured June performance from the
broader generalization judgment: one future-month holdout is strong evidence,
but ongoing monthly backtesting and drift monitoring are still required before
claiming consistent production accuracy.

In [12]:
best_booster = (
    test_metrics.loc[test_metrics["role"].eq("tuned gradient booster")]
    .sort_values(["test_mdape", "test_r2"], ascending=[True, False])
    .iloc[0]
)
random_forest_result = test_metrics.loc[
    test_metrics["family"].eq("Random Forest")
].iloc[0]
best_window_row = window_validation.sort_values(
    ["validation_mdape", "validation_r2"],
    ascending=[True, False],
).iloc[0]


def pct(value):
    return f"{value:.1%}"


def dollars(value):
    return f"${value:,.0f}"


report_text = f"""# Week 7 Advanced Models Weekly Report

## Evaluation design

The Week 3 preprocessing and Week 6 feature-engineering pipelines were rerun
from the monthly CSV files after shifting the timeline forward one month.
Training uses {TRAIN_START_MONTH} through {TRAIN_END_MONTH} ({len(train):,}
rows after train-only target handling), and the latest month, {TEST_MONTH},
is an untouched test set of {len(test):,} California single-family residence
sales. Listing-process leakage fields were excluded. Hyperparameters were
selected with complete-month chronological validation ending before the test
month.

## Gradient boosting comparison

The selected XGBoost and LightGBM configurations were chosen by mean validation
MdAPE, with validation R2 as a tie-breaker. On June 2026, the strongest booster
was **{best_booster['model']}**: R2 {best_booster['test_r2']:.3f}, MAE
{dollars(best_booster['test_mae'])}, RMSE {dollars(best_booster['test_rmse'])},
MAPE {pct(best_booster['test_mape'])}, and MdAPE
{pct(best_booster['test_mdape'])}. Its training R2 was
{best_booster['train_r2']:.3f}.

XGBoost and LightGBM both learn nonlinear interactions and additive corrections
to prior trees. XGBoost grows regularized trees level-wise and is deliberately
conservative here through row/column subsampling and L2 regularization.
LightGBM's leaf-wise histogram growth is faster and can capture sharp local
interactions, but it can overfit without depth, leaf, and child-size controls.
The reported validation dispersion and train/test gap therefore matter as much
as a small metric difference.

## Training-window finding

No May-only proxy window dominated every metric. The
{int(best_window_row['training_window_months'])}-month window had the lowest
MdAPE ({pct(best_window_row['validation_mdape'])}); nine months had the lowest
MAE; and eleven months had the best R2 and RMSE. The final fit uses the
assignment's required twelve-month shifted window, extending the strongest
variance/tail-risk proxy to a full seasonal cycle and the largest recent sample
without using any June outcome. Window length should be rechecked in a
multi-month rolling backtest as more monthly extracts arrive.

## Comparison with models explored in prior weeks

For a fair same-month comparison, the Week 6 Random Forest configuration was
retrained on the shifted W3/W6 matrix. Its June metrics were R2
{random_forest_result['test_r2']:.3f}, MAE
{dollars(random_forest_result['test_mae'])}, MAPE
{pct(random_forest_result['test_mape'])}, and MdAPE
{pct(random_forest_result['test_mdape'])}. Historical Week 4-6 tables used May,
so they are included for lineage but not treated as an apples-to-apples ranking.

Based on the latest same-month comparison, **{best_booster['model']}** is the
best overall statewide production candidate among the explored model families:
it leads on R2, RMSE, and narrowly on MAE, and its June result closely matches
its rolling validation behavior. The Random Forest remains better for typical
percentage error and should be retained as a benchmark or evaluated as an
ensemble component. This is a deliberate tradeoff, not a claim that XGBoost
wins every metric.

## Generalizability interpretation

The recommendation is evidence-based but conditional. A single June holdout
simulates the next production month and is more honest than a random split, yet
it cannot prove consistent accuracy for every future California home. The model
should be promoted with monthly rolling backtests, county/ZIP and price-band
monitoring, prediction-interval or uncertainty work, and retraining/drift
triggers. Unseen ZIP/district categories are safely routed by the encoder, but
localized sparse markets and luxury homes remain the highest-risk segments.
"""

report_path = OUTPUT_DIR / "Week7_Advanced_Models_Weekly_Report.md"
report_path.write_text(report_text, encoding="utf-8")

file_explanations = """# W7 File Explanations

- `05_advanced_models.ipynb` — Main executed deliverable. Imports the shifted
  raw CSVs, applies W3 preprocessing and W6 feature engineering, performs
  chronological tuning, reports June 2026 test metrics, and interprets results.
- `05_advanced_models.py` — Jupytext percent-format source paired with the
  notebook so the full workflow is easy to diff, rerun, and maintain.
- `advanced_model_cv_fold_plan.csv` — Complete-month rolling-origin train and
  validation periods used during hyperparameter tuning.
- `advanced_model_cv_detail_results.csv` — Per-fold metrics for every XGBoost
  and LightGBM candidate.
- `advanced_model_validation_results.csv` — Aggregated tuning metrics and the
  selected configuration for each gradient boosting family.
- `training_window_validation.csv` — May 2026 sensitivity check for recent
  3-, 6-, 9-, and 11-month proxy training windows.
- `advanced_model_test_metrics.csv` — Headline June 2026 metrics for selected
  XGBoost, selected LightGBM, and the shifted Week 6 Random Forest reference.
- `advanced_model_test_predictions.csv.gz` — Row-level June actual values,
  predictions, absolute errors, and percentage errors; gzip-compressed.
- `advanced_model_price_band_metrics.csv` — June metrics by actual-price
  quintile for segment-level error analysis.
- `advanced_model_feature_importance.csv` — Top 25 encoded features per final
  model, for diagnostic ranking rather than causal interpretation.
- `all_weeks_model_comparison.csv` — Week 4–7 metric lineage with a comparison
  basis flag that distinguishes historical May results from shifted June
  results.
- `Week7_Advanced_Models_Weekly_Report.md` — Short comparison, recommendation,
  limitations, and generalization interpretation.
- `FILE_EXPLANATIONS.md` — This concise inventory and purpose statement for
  every file in the W7 deliverable folder.
- `requirements.txt` — Pinned packages needed to reproduce the W7 workflow.
"""
file_explanations_path = OUTPUT_DIR / "FILE_EXPLANATIONS.md"
file_explanations_path.write_text(
    file_explanations,
    encoding="utf-8",
)

print(report_text)
print(f"Saved weekly report: {report_path}")
print(f"Saved file guide: {file_explanations_path}")

# Week 7 Advanced Models Weekly Report

## Evaluation design

The Week 3 preprocessing and Week 6 feature-engineering pipelines were rerun
from the monthly CSV files after shifting the timeline forward one month.
Training uses 2025-06 through 2026-05 (130,060
rows after train-only target handling), and the latest month, 2026-06,
is an untouched test set of 12,857 California single-family residence
sales. Listing-process leakage fields were excluded. Hyperparameters were
selected with complete-month chronological validation ending before the test
month.

## Gradient boosting comparison

The selected XGBoost and LightGBM configurations were chosen by mean validation
MdAPE, with validation R2 as a tie-breaker. On June 2026, the strongest booster
was **XGBoost (depth 8, lr 0.05, 500 trees)**: R2 0.812, MAE
$232,373, RMSE $666,903,
MAPE 25.2%, and MdAPE
11.8%. Its training R2 was
0.935.

XGBoost and LightGBM both learn nonlinear interactions and additive corrections
to prior trees. XGBoost 

## Completion checklist

- Shifted source months imported from `raw data` with pandas.
- Training fixed to June 2025–May 2026; untouched test fixed to June 2026.
- Week 3 cleaning and Week 6 feature engineering reproduced.
- XGBoost and LightGBM lightly tuned on depth, learning rate, and estimators.
- June test metrics stored as notebook tables and CSV artifacts.
- Same-month Random Forest reference and historical lineage included.
- Weekly report and per-file explanations saved in the W7 folder.